# Full OCR Run on Google Colab

Use this notebook when the full OCR run should happen in Colab and the local machine should only rebuild outputs with `--skip-ocr`.

Recommended runtime: **Python 3 + A100 GPU + High-RAM**. This notebook installs PaddleOCR only and disables EasyOCR fallback to avoid CUDA package conflicts in Colab.

For speed, the notebook copies the repo to `/content/ProjectDSDE_election2026` and does OCR on Colab's local disk, then writes the final artifact zip back to Google Drive. This avoids slow per-page reads/writes on Google Drive.

Default mode is a fresh accuracy run: local scratch disk, no EasyOCR fallback, no per-batch report rebuild, no overwrite of already-created JSON, explicit `gpu:0`, DPI 300, larger detection side limit, and full zone OCR (`metadata + summary + table`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import shutil

DRIVE_REPO_DIR = Path('/content/drive/MyDrive/election2026/ProjectDSDE_election2026')
WORK_DIR = Path('/content/ProjectDSDE_election2026')
PDF_ZIP = Path('/content/drive/MyDrive/election2026/raw_pdfs.zip')
ARTIFACT_DIR = Path('/content/drive/MyDrive/election2026/artifacts')
CONFIG = 'configs/chaiyaphum_2.yaml'

# Fast fresh run defaults. For a future resume run, set START_FRESH = False
# and RESTORE_ARTIFACT_ZIP to the checkpoint/final artifact zip you want to continue from.
USE_LOCAL_SCRATCH = True
START_FRESH = True
RESTORE_ARTIFACT_ZIP = None  # Example: ARTIFACT_DIR / 'ocr_checkpoint_latest.zip'
SPEED_PROFILE = 'accuracy'  # accuracy uses dpi=300 and metadata+summary+table OCR

if USE_LOCAL_SCRATCH:
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    ignore = shutil.ignore_patterns(
        '.git', '__pycache__', '.pytest_cache', '.venv', '.cache',
        'data/raw/images', 'data/raw/ocr', 'data/raw/pdfs',
        'data/processed/parsed', 'outputs'
    )
    shutil.copytree(DRIVE_REPO_DIR, WORK_DIR, ignore=ignore)
    os.chdir(WORK_DIR)
else:
    os.chdir(DRIVE_REPO_DIR)

print('cwd:', Path.cwd())
print('using local scratch:', USE_LOCAL_SCRATCH)
print('start fresh:', START_FRESH)
print('speed profile:', SPEED_PROFILE)
print('config exists:', Path(CONFIG).exists())
print('pdf zip exists:', PDF_ZIP.exists(), PDF_ZIP)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install PaddleOCR-Only Environment

Run this once per fresh runtime. If Colab asks to restart the runtime after install, restart, then rerun only the mount/setup cell above and continue from the import check cell.

In [ ]:
import subprocess

def run(cmd):
    print('\n$', ' '.join(cmd))
    subprocess.run(cmd, check=True)

run(['python', '-m', 'pip', 'install', '-U', 'pip', 'setuptools', 'wheel'])
run(['python', '-m', 'pip', 'uninstall', '-y', 'easyocr', 'torch', 'torchvision', 'paddlepaddle', 'paddlepaddle-gpu', 'paddleocr'])
run(['python', '-m', 'pip', 'install', 'paddlepaddle-gpu==3.2.0', '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'])
run(['python', '-m', 'pip', 'install', 'paddleocr'])
run(['python', '-m', 'pip', 'install', 'langchain-text-splitters'])
run([
    'python', '-m', 'pip', 'install',
    'pandas>=2.2', 'pyyaml>=6.0', 'pymupdf>=1.24', 'pillow>=10.0',
    'numpy>=1.26', 'matplotlib>=3.8', 'streamlit>=1.34', 'altair>=5.0',
    'pyarrow>=15.0', 'pytest>=8.0'
])

## 3. Verify Imports and Apply Fast OCR Config

In [ ]:
import subprocess
import yaml
subprocess.run([
    'python', '-c',
    "import paddle; import paddleocr; print('paddle:', paddle.__version__); print('device:', paddle.device.get_device()); print('paddleocr import: ok')"
], check=True)

from pathlib import Path
config_path = Path(CONFIG)
config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
ocr_config = config.setdefault('ocr', {})
ocr_config['fallback_engine'] = ''
paddle_config = ocr_config.setdefault('paddle', {})
paddle_config['device'] = 'gpu:0'
paddle_config['enable_hpi'] = True
paddle_config['use_tensorrt'] = False
paddle_config['precision'] = 'fp32'
zone_ocr = ocr_config.setdefault('zone_ocr', {})
zone_ocr['enabled'] = True
if SPEED_PROFILE == 'fast_safe':
    ocr_config['dpi'] = 250
    zone_ocr['zones'] = ['summary', 'table']
    paddle_config['text_det_limit_side_len'] = 1280
elif SPEED_PROFILE == 'accuracy':
    ocr_config['dpi'] = 300
    zone_ocr['zones'] = ['metadata', 'summary', 'table']
    paddle_config['text_det_limit_side_len'] = 1920
else:
    raise ValueError(f'Unknown SPEED_PROFILE: {SPEED_PROFILE}')
config_path.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding='utf-8')
print('Applied OCR config:', {
    'fallback_engine': ocr_config.get('fallback_engine'),
    'dpi': ocr_config.get('dpi'),
    'zone_ocr': zone_ocr,
    'paddle': paddle_config,
})

## 4. Extract Raw PDF Zip

Expected zip path: `/content/drive/MyDrive/election2026/raw_pdfs.zip`.

After extraction the folder must contain `data/raw/pdfs/extracted/เขตเลือกตั้งที่ 2/`.

In [ ]:
from pathlib import Path
import shutil
import zipfile

pdf_extract_root = Path('data/raw/pdfs/extracted')
target_dir = pdf_extract_root / 'เขตเลือกตั้งที่ 2'
pdf_extract_root.mkdir(parents=True, exist_ok=True)

if not PDF_ZIP.exists():
    raise FileNotFoundError(f'Missing raw PDF zip: {PDF_ZIP}')

print('extracting:', PDF_ZIP)
with zipfile.ZipFile(PDF_ZIP) as archive:
    archive.extractall(pdf_extract_root)

if not target_dir.exists():
    candidates = [p for p in pdf_extract_root.rglob('เขตเลือกตั้งที่ 2') if p.is_dir()]
    if candidates:
        print('copying nested district folder:', candidates[0], '->', target_dir)
        shutil.copytree(candidates[0], target_dir, dirs_exist_ok=True)

pdfs = list(pdf_extract_root.rglob('*.pdf')) + list(pdf_extract_root.rglob('*.PDF'))
print('pdf files found:', len(pdfs))
for sample in pdfs[:10]:
    print(sample)

if len(pdfs) < 35:
    raise RuntimeError('Expected at least 35 PDF files. Check PDF_ZIP or extraction path.')

## 5. Fresh Run Cleanup or Resume Restore

With `START_FRESH = True`, this removes old local/failed OCR outputs so this run starts fresh.

For a future resume run, set `START_FRESH = False` and set `RESTORE_ARTIFACT_ZIP` in the first cell.

In [ ]:
from pathlib import Path
import shutil
import zipfile

if START_FRESH:
    for folder in ['data/raw/ocr', 'data/raw/images', 'data/processed/parsed']:
        path = Path(folder)
        if path.exists():
            print('removing folder:', path)
            shutil.rmtree(path)
elif RESTORE_ARTIFACT_ZIP:
    restore_path = Path(RESTORE_ARTIFACT_ZIP)
    if not restore_path.exists():
        raise FileNotFoundError(f'Missing restore artifact: {restore_path}')
    print('restoring artifact:', restore_path)
    with zipfile.ZipFile(restore_path) as archive:
        archive.extractall(Path.cwd())
else:
    print('Resume mode without restore artifact. Existing files in local scratch will be reused if present.')

for pattern in [
    'data/processed/election_results_long.csv',
    'data/processed/polling_station_summary.csv',
    'data/processed/constituency_votes.csv',
    'data/processed/partylist_votes.csv',
    'data/processed/dashboard_dataset.parquet',
    'data/processed/dashboard_dataset.csv',
    'data/processed/validation_report.csv',
    'data/processed/ocr_progress.csv',
    'data/processed/review_queue.csv',
    'data/processed/accuracy_report.csv',
    'data/processed/accuracy_details.csv',
]:
    path = Path(pattern)
    if START_FRESH and path.exists():
        print('removing file:', path)
        path.unlink()

## 6. Check Manifest Before OCR

Do not continue if `pdf_exists` is false or `missing_pdf` appears.

In [ ]:
import pandas as pd
import subprocess

subprocess.run(['python', '-m', 'src.pipeline.ocr_progress', '--config', CONFIG], check=True)
progress = pd.read_csv('data/processed/ocr_progress.csv', encoding='utf-8-sig')
display(progress[['manifest_index', 'form_type', 'file_name', 'pdf_exists', 'pdf_pages', 'status']])
print('status counts:', progress['status'].value_counts(dropna=False).to_dict())
print('pdf pages total:', int(progress['pdf_pages'].fillna(0).sum()))

if not progress['pdf_exists'].all():
    raise RuntimeError('Some PDFs are missing. Fix PDF extraction before running OCR.')

## 7. Run Full OCR Batches

This can take a long time. If Colab stops, rerun from this cell and change `BATCHES` to only the remaining ranges.

Default speed settings: `OVERWRITE = False` so completed raw JSON is reused, and `RUN_REPORTS_AFTER_EACH_BATCH = False` so heavy report generation only happens at the end. With a clean scratch directory, `OVERWRITE = False` still starts fresh because there is no old OCR JSON to reuse.

In [ ]:
import subprocess
import shutil
import pandas as pd
from pathlib import Path

BATCHES = [(1, 4), (5, 8), (9, 16), (17, 24), (25, 32), (33, 35)]
OVERWRITE = False
RUN_REPORTS_AFTER_EACH_BATCH = False
CHECKPOINT_EACH_BATCH = False  # fastest. Set True if the runtime is unstable.

def run_cmd(cmd):
    print('\n$', ' '.join(cmd))
    subprocess.run(cmd, check=True)

for start, end in BATCHES:
    cmd = ['python', '-m', 'src.ocr.extract', '--config', CONFIG, '--start-index', str(start), '--end-index', str(end)]
    if OVERWRITE:
        cmd.append('--overwrite')
    run_cmd(cmd)
    if RUN_REPORTS_AFTER_EACH_BATCH:
        run_cmd(['python', '-m', 'src.pipeline.run_all', '--config', CONFIG, '--skip-ocr'])
    run_cmd(['python', '-m', 'src.pipeline.ocr_progress', '--config', CONFIG])
    progress = pd.read_csv('data/processed/ocr_progress.csv', encoding='utf-8-sig')
    print('after batch', start, end, progress['status'].value_counts(dropna=False).to_dict())
    if CHECKPOINT_EACH_BATCH:
        checkpoint_path = ARTIFACT_DIR / f'ocr_checkpoint_{start}_{end}.zip'
        latest_path = ARTIFACT_DIR / 'ocr_checkpoint_latest.zip'
        for path in [checkpoint_path, latest_path]:
            if path.exists():
                path.unlink()
        checkpoint_cmd = ['zip', '-qr', str(checkpoint_path), 'data/raw/ocr', 'data/processed/parsed', 'data/processed/ocr_progress.csv']
        run_cmd(checkpoint_cmd)
        shutil.copy2(checkpoint_path, latest_path)
        print('checkpoint saved:', checkpoint_path)

print('Full OCR batch loop finished.')

## 8. Final Reports

In [ ]:
import subprocess
import pandas as pd

for cmd in [
    ['python', '-m', 'src.pipeline.run_all', '--config', CONFIG, '--skip-ocr'],
    ['python', '-m', 'src.quality.review_queue', '--config', CONFIG],
    ['python', '-m', 'src.quality.evaluate_accuracy', '--config', CONFIG],
    ['python', '-m', 'src.pipeline.ocr_progress', '--config', CONFIG],
]:
    print('\n$', ' '.join(cmd))
    subprocess.run(cmd, check=True)

for path in [
    'data/processed/ocr_progress.csv',
    'data/processed/validation_report.csv',
    'data/processed/review_queue.csv',
    'data/processed/accuracy_report.csv',
    'data/processed/constituency_votes.csv',
    'data/processed/partylist_votes.csv',
]:
    df = pd.read_csv(path, encoding='utf-8-sig')
    print('\n', path, df.shape)
    display(df.head(20))

## 9. Zip Artifacts Back to Drive

The zip should be much larger than a few KB because it must include raw OCR JSON files.

In [ ]:
import subprocess
from pathlib import Path

artifact_path = ARTIFACT_DIR / 'ocr_all_fresh_artifacts_v2.zip'
if artifact_path.exists():
    artifact_path.unlink()

zip_items = [
    'data/raw/ocr',
    'data/processed/parsed',
    'data/processed/election_results_long.csv',
    'data/processed/polling_station_summary.csv',
    'data/processed/constituency_votes.csv',
    'data/processed/partylist_votes.csv',
    'data/processed/dashboard_dataset.parquet',
    'data/processed/dashboard_dataset.csv',
    'data/processed/validation_report.csv',
    'data/processed/ocr_progress.csv',
    'data/processed/review_queue.csv',
    'data/processed/accuracy_report.csv',
    'data/processed/accuracy_details.csv',
    'outputs/reports',
]

cmd = ['zip', '-r', str(artifact_path)] + zip_items
print('$', ' '.join(cmd))
subprocess.run(cmd, check=True)

size_mb = artifact_path.stat().st_size / (1024 * 1024)
raw_json_count = len(list(Path('data/raw/ocr').rglob('*.json')))
parsed_csv_count = len(list(Path('data/processed/parsed').rglob('*.csv')))
print('artifact:', artifact_path)
print('artifact size MB:', round(size_mb, 2))
print('raw OCR JSON files:', raw_json_count)
print('parsed CSV files:', parsed_csv_count)

if raw_json_count == 0:
    raise RuntimeError('No raw OCR JSON files were zipped. OCR did not run correctly.')

## 10. Local Restore Command

After downloading `ocr_all_fresh_artifacts_v2.zip` to the local machine, run:

```bash
cd /Users/pornmongkol/Documents/ProjectDSDE_election2026
unzip -o ~/Downloads/ocr_all_fresh_artifacts_v2.zip
.venv/bin/python -m src.pipeline.run_all --config configs/chaiyaphum_2.yaml --skip-ocr
.venv/bin/python -m src.quality.evaluate_accuracy --config configs/chaiyaphum_2.yaml
streamlit run src/dashboard/app.py
```